In [ ]:
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import datetime as dt
import dask.dataframe as dd

import matplotlib.pyplot as plt
from matplotlib import colors
import soundfile as sf
import pyaudio
import matplotlib.patches as patches
from pathlib import Path
from sklearn.cluster import KMeans
import fsspec

In [ ]:
def list_audio_devices():
    """List all available audio devices."""
    p = pyaudio.PyAudio()
    print("Available audio devices:")
    for i in range(p.get_device_count()):
        device_info = p.get_device_info_by_index(i)
        print(f"Device {i}: {device_info['name']} (Sample Rate: {device_info['defaultSampleRate']})")
    p.terminate()

# Call this function once to see the available devices
list_audio_devices()

In [ ]:
import sys

sys.path.append("../src")
sys.path.append("../src/bout")

In [ ]:
from core import SITE_NAMES, FREQUENCY_COLOR_MAPPINGS
import bout.clustering as clstr
import bout.assembly as bt
import plot as bt_plt
import activity.subsampling as ss
import activity.activity_assembly as actvt

from cli import get_file_paths
from calls import plot_call_features, compute_features, call_extraction

In [ ]:
FREQUENCY_COLOR_MAPPINGS = {
                    'LF' : 'cyan',
                    'HF' : 'orange'
                        }

LABEL_FOR_GROUPS = {
                    0: 'LF', 
                    1: 'HF'
                    }

FIGSIZE = (12, 6)

DURATION = 300

In [ ]:
PADDED_CALL_LENGTH = 0.06

def open_and_get_call_info(audio_file, dets):
    welch_key = 'all_locations'
    output_dir = Path(f'../data/generated_welch/{welch_key}')
    output_file_type = 'top1_inbouts_welch_signals'
    welch_data = pd.read_csv(output_dir / f'2022_{welch_key}_{output_file_type}.csv', index_col=0, low_memory=False)
    k = 2
    kmean_welch = KMeans(n_clusters=k, n_init=10, random_state=1).fit(welch_data.values)

    features_of_interest = gather_features_of_interest(dets, kmean_welch, audio_file)

    dets.reset_index(drop=True, inplace=True)
    dets['index'] = dets.index
    dets['file_name'] = pd.DatetimeIndex(pd.to_datetime(dets['input_file'], format='%Y%m%d_%H%M%S', exact=False)).strftime('%Y%m%d_%H%M%S.WAV')
    dets['sampling_rate'] = len(dets) * [audio_file.samplerate]
    dets.insert(0, 'SNR', features_of_interest['snrs'])
    dets.insert(0, 'peak_frequency', features_of_interest['peak_freqs'])
    dets.insert(0, 'KMEANS_CLASSES', pd.Series(features_of_interest['classes']).map(LABEL_FOR_GROUPS))

    return features_of_interest['call_signals'], dets

def gather_features_of_interest(dets, kmean_welch, audio_file):
    fs = audio_file.samplerate
    features_of_interest = dict()
    features_of_interest['call_signals'] = []
    features_of_interest['welch_signals'] = []
    features_of_interest['snrs'] = []
    features_of_interest['peak_freqs'] = []
    features_of_interest['classes'] = []
    nyquist = fs//2
    for index, row in dets.iterrows():
        audio_seg, length_of_section = get_section_of_call_in_file(row, audio_file)
        
        freq_pad = 2000
        low_freq_cutoff = row['low_freq']-freq_pad
        high_freq_cutoff = min(nyquist-1, row['high_freq']+freq_pad)
        band_limited_audio_seg = call_extraction.bandpass_audio_signal(audio_seg, fs, low_freq_cutoff, high_freq_cutoff)

        signal = band_limited_audio_seg.copy()
        signal[:int(fs*(length_of_section))] = 0
        noise = band_limited_audio_seg - signal
        snr_call_signal = signal[-int(fs*length_of_section):]
        snr_noise_signal = noise[:int(fs*length_of_section)]
        features_of_interest['call_signals'].append(snr_call_signal)

        snr = call_extraction.get_snr_from_band_limited_signal(snr_call_signal, snr_noise_signal)
        features_of_interest['snrs'].append(snr)

        welch_info = dict()
        welch_info['num_points'] = 100
        max_visible_frequency = 96000
        welch_info['max_freq_visible'] = max_visible_frequency
        welch_signal = compute_features.compute_welch_psd_of_call(snr_call_signal, fs, welch_info)
        features_of_interest['welch_signals'].append(welch_signal)

        peaks = np.where(welch_signal==max(welch_signal))[0][0]
        features_of_interest['peak_freqs'].append((max_visible_frequency/len(welch_signal))*peaks)
        
        welch_signal = (welch_signal).reshape(1, len(welch_signal))
        features_of_interest['classes'].append(kmean_welch.predict(welch_signal)[0])

    features_of_interest['call_signals'] = np.array(features_of_interest['call_signals'], dtype='object')

    return features_of_interest

def get_section_of_call_in_file(detection, audio_file):
    fs = audio_file.samplerate
    call_dur = (detection['end_time'] - detection['start_time'])
    pad = 0.004
    start = detection['start_time'] - call_dur - (3*pad)
    duration = (2 * call_dur) + (4*pad)
    end = detection['end_time']
    audio_file.seek(int(fs*start))
    audio_seg = audio_file.read(int(fs*duration))

    length_of_section = call_dur + (2*pad)

    return audio_seg, length_of_section

In [ ]:
def plot_colored_dets_over_audio_seg(audio_features, spec_features, plot_dets):
    """
    Function to plot the spectrogram of a provided audio segment with overlayed detections
    """

    audio_seg = audio_features['audio_seg']
    fs = audio_features['sample_rate']
    start = audio_features['start']
    duration = audio_features['duration']

    plt.figure(figsize=FIGSIZE)
    plt.rcParams.update({'font.size': 24})
    plt.title(f"BatDetect2 detections on {audio_features['file_path'].name}", fontsize=22)
    plt.specgram(audio_seg, NFFT=spec_features['NFFT'], cmap=spec_features['cmap'], vmin=spec_features['vmin'])

    yellow_patch = patches.Patch(facecolor='yellow', edgecolor='k', label='Detections')

    legend_patches = [yellow_patch]
    ax = plt.gca()
    for i, row in plot_dets.iterrows():
        rect = patches.Rectangle(((row['start_time'] - start)*(fs/2), row['low_freq']/(fs/2)), 
                        (row['end_time'] - row['start_time'])*(fs/2), (row['high_freq'] - row['low_freq'])/(fs/2), 
                        linewidth=2, edgecolor=FREQUENCY_COLOR_MAPPINGS[row['KMEANS_CLASSES']], facecolor='none', alpha=1)
        
        ax.add_patch(rect)
    plt.yticks(ticks=np.linspace(0, 96000/(fs/2), 11), labels=np.linspace(0, 96, 11).astype('int'))
    plt.xticks(ticks=np.linspace(0, duration*(fs/2), 11), labels=np.round(np.linspace(start, start+duration, 11, dtype='float'), 2), rotation=30)
    plt.ylabel("Frequency (kHz)")
    plt.ylim(0, 96000/(fs/2))
    plt.xlabel("Time (s)")
    plt.gcf().autofmt_xdate()
    plt.grid(axis='y')
    plt.legend(handles=legend_patches, fontsize=20, ncol=int(len(legend_patches)**0.5), loc='upper right')

    plt.tight_layout()
    plt.show()


def plot_colored_dets_over_audio_seg_w_bounds(audio_features, spec_features, call_features, plot_dets):
    """
    Function to plot the spectrogram of a provided audio segment with overlayed detections
    """

    audio_seg = audio_features['audio_seg']
    fs = audio_features['sample_rate']
    start = audio_features['start']
    duration = audio_features['duration']

    plt.figure(figsize=FIGSIZE)
    plt.rcParams.update({'font.size': 24})
    plt.title(f"BatDetect2 detections on {audio_features['file_path'].name}", fontsize=22)
    plt.specgram(audio_seg, NFFT=spec_features['NFFT'], cmap=spec_features['cmap'], vmin=spec_features['vmin'])

    yellow_patch = patches.Patch(facecolor='yellow', edgecolor='k', label='Detections')

    legend_patches = [yellow_patch]
    ax = plt.gca()
    for i, row in plot_dets.iterrows():
        rect = patches.Rectangle(((row['start_time'] - start)*(fs/2), row['low_freq']/(fs/2)), 
                        (row['end_time'] - row['start_time'])*(fs/2), (row['high_freq'] - row['low_freq'])/(fs/2), 
                        linewidth=2, edgecolor=FREQUENCY_COLOR_MAPPINGS[row['KMEANS_CLASSES']], facecolor='none', alpha=1)
        
        ax.add_patch(rect)

    rect = patches.Rectangle((0,(call_features['median_lf']-7000)/(fs/2)), duration*fs, 14000/(fs/2), 
                             linewidth=2, edgecolor=FREQUENCY_COLOR_MAPPINGS['LF'], facecolor=FREQUENCY_COLOR_MAPPINGS['LF'], alpha=0.4)
    ax.add_patch(rect)
    rect = patches.Rectangle((0,(call_features['median_hf']-7000)/(fs/2)), duration*fs, (fs/2 - (call_features['median_hf']-7000))/(fs/2), 
                             linewidth=2, edgecolor=FREQUENCY_COLOR_MAPPINGS['HF'], facecolor=FREQUENCY_COLOR_MAPPINGS['HF'], alpha=0.4)
    ax.add_patch(rect)

    plt.yticks(ticks=np.linspace(0, 96000/(fs/2), 11), labels=np.linspace(0, 96, 11).astype('int'))
    plt.xticks(ticks=np.linspace(0, duration*(fs/2), 11), labels=np.round(np.linspace(start, start+duration, 11, dtype='float'), 2), rotation=30)
    plt.ylabel("Frequency (kHz)")
    plt.ylim(0, 96000/(fs/2))
    plt.xlabel("Time (s)")
    plt.gcf().autofmt_xdate()
    plt.grid(axis='y')
    plt.legend(handles=legend_patches, fontsize=20, ncol=int(len(legend_patches)**0.5), loc='upper right')

    plt.tight_layout()
    plt.show()


def plot_audio_seg(audio_features, spec_features):
    """
    Function to plot the spectrogram of a provided audio segment
    """

    audio_seg = audio_features['audio_seg']
    fs = audio_features['sample_rate']
    start = audio_features['start']
    duration = audio_features['duration']

    plt.figure(figsize=FIGSIZE)
    plt.rcParams.update({'font.size' : 24})
    plt.title(f"Audio collected from {audio_features['site_name']} on {audio_features['file_datetime']} UTC", fontsize=22)
    plt.rcParams.update({'font.size': 24})
    plt.specgram(audio_seg, NFFT=spec_features['NFFT'], cmap=spec_features['cmap'], vmin=spec_features['vmin'])
    plt.yticks(ticks=np.linspace(0, 96000/(fs/2), 11), labels=np.linspace(0, 96, 11).astype('int'))
    plt.xticks(ticks=np.linspace(0, duration*(fs/2), 11), labels=np.round(np.linspace(start, start+duration, 11, dtype='float'), 2), rotation=30)
    plt.ylabel("Frequency (kHz)")
    plt.ylim(0, 96000/(fs/2))
    plt.xlabel("Time (s)")
    plt.gcf().autofmt_xdate()
    plt.grid(axis='y')

    plt.tight_layout()
    plt.show()

def load_and_plot_all_examples_file(file_path, bd2_dets, start, duration, rm_dB=50, nfft=1024):
    audio_file = sf.SoundFile(file_path)
    fs = audio_file.samplerate
    audio_file.seek(int(fs*start))
    audio_seg = audio_file.read(int(fs*duration))
    vmin = 20*np.log10(np.max(audio_seg)) -  rm_dB # hide anything below -rm_dB dB

    audio_features = dict()
    audio_features['site_name'] = SITE_NAMES[file_path.parent.name]
    audio_features['file_datetime'] = dt.datetime.strptime(file_path.name, "%Y%m%d_%H%M%S.WAV").strftime('%Y/%m/%d %H:%M')
    audio_features['file_path'] = file_path
    audio_features['audio_seg'] = audio_seg
    audio_features['sample_rate'] = fs
    audio_features['start'] = start
    audio_features['duration'] = duration

    spec_features = dict()
    spec_features['vmin'] = vmin
    spec_features['NFFT'] = nfft
    spec_features['cmap'] = 'jet'

    site_key = file_path.parent.name

    plot_audio_seg(audio_features, spec_features)
    call_signals, dets = open_and_get_call_info(audio_file, bd2_dets.copy())
    plot_dets = dets.loc[(dets['start_time'] > start)&(dets['end_time'] < (start+duration))]
    plot_colored_dets_over_audio_seg(audio_features, spec_features, plot_dets)

    median_peak_HF_freq = dets[dets['KMEANS_CLASSES']=='HF']['peak_frequency'].median()
    median_peak_LF_freq = dets[dets['KMEANS_CLASSES']=='LF']['peak_frequency'].median()
    print(f'Median LF Frequency in file: {median_peak_LF_freq}')
    print(f'Median HF Frequency in file: {median_peak_HF_freq}')
    lf_inds = (plot_dets['peak_frequency']<median_peak_LF_freq+7000)&(plot_dets['peak_frequency']>median_peak_LF_freq-7000)
    hf_inds = (plot_dets['peak_frequency']>median_peak_HF_freq-7000)

    lf_dets = plot_dets[lf_inds&(plot_dets['KMEANS_CLASSES']=='LF')]
    hf_dets = plot_dets[hf_inds&(plot_dets['KMEANS_CLASSES']=='HF')]

    all_dets = pd.concat([hf_dets, lf_dets]).sort_index()

    call_features = dict()
    call_features['median_lf'] = median_peak_LF_freq
    call_features['median_hf'] = median_peak_HF_freq
    plot_colored_dets_over_audio_seg_w_bounds(audio_features, spec_features, call_features, all_dets)

    return plot_dets, all_dets

In [ ]:
site_key = 'Foliage'

fig_details = dict()
fig_details['site_name'] = SITE_NAMES[site_key]
print(f'Looking at {fig_details["site_name"]}')
files_from_loc = sorted(list(Path(f'../data/audiomoth_recordings/').glob(pattern=f'*/{site_key}/*.WAV')))

file_path = files_from_loc[0]
filename = file_path.name
csv_path = Path(f'../batdetect2_outputs/recover-20210912/{site_key}/bd2_{filename.split(".")[0]}.csv')
print(f'Looking at {file_path.name}')
start = 1439.5
duration = 1.5
rm_dB = 60
nfft = 512

data_params = dict()
data_params['site_tag'] = site_key
data_params['type_tag'] = ''
data_params['cur_dc_tag'] = '1800of1800'
batdetect2_predictions_no_dutycycle = actvt.assemble_single_bd2_output_use_thresholds_to_group(csv_path, data_params)

mis_dets, fix_dets = load_and_plot_all_examples_file(file_path, batdetect2_predictions_no_dutycycle, start, DURATION, rm_dB)

In [ ]:
removed_dets = mis_dets.loc[list(set(mis_dets.index) - set(fix_dets.index))]
removed_dets

In [ ]:
site_key = 'Foliage'

fig_details = dict()
fig_details['site_name'] = SITE_NAMES[site_key]
print(f'Looking at {fig_details["site_name"]}')
files_from_loc = sorted(list(Path(f'../data/audiomoth_recordings/').glob(pattern=f'*/{site_key}/*.WAV')))

file_path = files_from_loc[0]
filename = file_path.name
csv_path = Path(f'../batdetect2_outputs/recover-20210912/{site_key}/bd2_{filename.split(".")[0]}.csv')
print(f'Looking at {file_path.name}')
start = 1439.5
duration = 1.5
rm_dB = 60
nfft = 512

data_params = dict()
data_params['site_tag'] = site_key
data_params['type_tag'] = ''
data_params['cur_dc_tag'] = '1800of1800'
batdetect2_predictions_no_dutycycle = actvt.assemble_single_bd2_output_use_thresholds_to_group(csv_path, data_params)

mis_dets, fix_dets = load_and_plot_all_examples_file(file_path, batdetect2_predictions_no_dutycycle, start, duration, rm_dB)

In [ ]:
audio_file = sf.SoundFile(file_path)
fs = audio_file.samplerate
audio_file.seek(int(fs*start))
audio_seg = audio_file.read(int(fs*duration))
audio_seg

In [ ]:
import sounddevice as sd
import simpleaudio as sa

In [ ]:
sample_rate = 44100
frequency = 440

In [ ]:
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

# Generate sine wave
sine_wave = 0.5 * np.sin(2 * np.pi * frequency * t)

# Convert to 16-bit PCM format
audio_data = (sine_wave * 32767).astype(np.int16)

try:
    print(f"Playing a {frequency} Hz sine wave for {duration} seconds...")
    # Create a wave object
    wave_obj = sa.WaveObject(audio_data.tobytes(), num_channels=1, bytes_per_sample=2, sample_rate=sample_rate)
    
    # Play the wave
    play_obj = wave_obj.play()
    play_obj.wait_done()  # Wait until playback is finished
    print("Playback finished.")

except Exception as e:
    print(f"Error during playback: {e}")

In [ ]:
removed_dets = mis_dets.loc[list(set(mis_dets.index) - set(fix_dets.index))]
removed_dets

In [ ]:
site_key = 'Foliage'

fig_details = dict()
fig_details['site_name'] = SITE_NAMES[site_key]
print(f'Looking at {fig_details["site_name"]}')
files_from_loc = sorted(list(Path(f'../data/audiomoth_recordings/').glob(pattern=f'*/{site_key}/*.WAV')))

file_path = files_from_loc[0]
filename = file_path.name
csv_path = Path(f'../batdetect2_outputs/recover-20210912/{site_key}/bd2_{filename.split(".")[0]}.csv')
print(f'Looking at {file_path.name}')
start = 1459
duration = 1
rm_dB = 60
nfft = 512

data_params = dict()
data_params['site_tag'] = site_key
data_params['type_tag'] = ''
data_params['cur_dc_tag'] = '1800of1800'
batdetect2_predictions_no_dutycycle = actvt.assemble_single_bd2_output_use_thresholds_to_group(csv_path, data_params)

mis_dets, fix_dets = load_and_plot_all_examples_file(file_path, batdetect2_predictions_no_dutycycle, start, duration, rm_dB)

In [ ]:
list(set(mis_dets.index) - set(fix_dets.index))

In [ ]:
removed_dets = mis_dets.loc[list(set(mis_dets.index) - set(fix_dets.index))]
removed_dets

In [ ]:
data_params["site_name"] = 'Foliage'
data_params["site_tag"] = 'Foliage'
data_params["type_tag"] = ''
data_params["detector_tag"] = 'bd2'
data_params["assembly_type"] = 'thresh'

file_paths = get_file_paths(data_params)

location_df_thresh = pd.read_csv(f'{file_paths["SITE_folder"]}/{file_paths["detector_TYPE_SITE_YEAR"]}.csv', low_memory=False, index_col=0)
location_df_thresh

In [ ]:
selection_date = dt.datetime(2022,6,15,5,0,0)

In [ ]:
selected_group_thresh = location_df_thresh[pd.to_datetime(location_df_thresh['input_file'], format="%Y%m%d_%H%M%S.WAV", exact=False)<=selection_date]
selected_group_thresh

In [ ]:
def add_frequency_group_to_file_dets(file_dets, location_classes):
    file_classes = location_classes[pd.to_datetime(location_classes['file_name'], 
                                                   format='%Y%m%d_%H%M%S.WAV', exact=False)==file_dets.name].copy()

    file_dets.insert(0, 'index_in_summary', file_dets.index)
    file_dets.set_index('index_in_file', inplace=True)

    classified = file_classes['KMEANS_CLASSES']!=''
    file_classes.loc[classified, 'peak_frequency'] = file_classes.loc[classified, 'peak_frequency'].astype('float64')

    file_dets.insert(0, 'peak_frequency', [np.NaN]*len(file_dets))
    file_dets.loc[file_classes['index_in_file'], 'freq_group'] = file_classes['KMEANS_CLASSES'].values
    file_dets.loc[file_classes['index_in_file'], 'peak_frequency'] = file_classes['peak_frequency'].values

    # for group in ['LF', 'HF']:
    #     group_classified_dets = (file_dets['freq_group']==group)

    #     low_assert1 = (file_dets.loc[group_classified_dets, 'peak_frequency'] > (file_dets.loc[group_classified_dets, 'low_freq']).median()-4000)
    #     low_assert2 = (file_dets.loc[group_classified_dets, 'peak_frequency'] > (file_dets.loc[group_classified_dets, 'low_freq'])-4000)
    #     assert(low_assert1|low_assert2).all()
    #     high_assert1 = (file_dets.loc[group_classified_dets, 'peak_frequency'] < (file_dets.loc[group_classified_dets, 'high_freq']).median()+4000)
    #     high_assert2 = (file_dets.loc[group_classified_dets, 'peak_frequency'] < (file_dets.loc[group_classified_dets, 'high_freq'])+4000)
    #     assert(high_assert1|high_assert2).all()

    return file_dets

def add_frequency_groups_to_summary_using_kmeans(location_df, file_paths, data_params, save=True):
    location_df.insert(0, 'freq_group', '')
    location_classes = pd.read_csv(Path(file_paths['SITE_classes_file']), index_col=0)
    location_df.insert(0, 'input_file_dt', pd.to_datetime(location_df['input_file'], format='%Y%m%d_%H%M%S.WAV', exact=False))
    location_df_grouped = location_df.groupby('input_file_dt', group_keys=True)

    location_df_classified = location_df_grouped.apply(lambda x: add_frequency_group_to_file_dets(x, location_classes))

    location_df_only_classified = location_df_classified.loc[location_df_classified['freq_group']!='']
    location_df_only_classified = location_df_only_classified.droplevel(level=0)
    location_df_only_classified = location_df_only_classified.reset_index()

    if data_params['type_tag'] != '':
        location_df_only_classified = location_df_only_classified.loc[location_df_only_classified['freq_group']==data_params['type_tag']]

    if save:
        location_df_only_classified.to_csv(f'{file_paths["SITE_folder"]}/{file_paths["detector_TYPE_SITE_YEAR"]}.csv')

    return location_df_only_classified

In [ ]:
data_params["site_name"] = 'Foliage'
data_params["site_tag"] = 'Foliage'
data_params["type_tag"] = ''
data_params["detector_tag"] = 'bd2'
data_params["assembly_type"] = 'kmeans'

file_paths = get_file_paths(data_params)
file_paths['SITE_classes_file'] = f"{file_paths['SITE_classes_file'][:-4]}_raw.csv"

init_location_sum = actvt.assemble_initial_location_summary(file_paths) 
init_location_sum.reset_index(inplace=True)
init_location_sum.rename({'index':'index_in_file'}, axis='columns', inplace=True)
location_df_kmeans_raw = add_frequency_groups_to_summary_using_kmeans(init_location_sum.copy(), file_paths, data_params, save=False)

In [ ]:
location_df_kmeans_raw

In [ ]:
data_params["site_name"] = 'Foliage'
data_params["site_tag"] = 'Foliage'
data_params["type_tag"] = ''
data_params["detector_tag"] = 'bd2'
data_params["assembly_type"] = 'kmeans'
data_params['recording_start'] = '00:00'
data_params['recording_end'] = '16:00'
data_params['cur_dc_tag'] = '30of30'
data_params['cycle_length'] = int(data_params['cur_dc_tag'].split('of')[-1])
data_params['time_on'] = int(data_params['cur_dc_tag'].split('of')[0])
data_params['time_on_in_secs'] = 60*data_params['time_on']

file_paths = get_file_paths(data_params)

location_df_kmeans = pd.read_csv(f'{file_paths["SITE_folder"]}/{file_paths["detector_TYPE_SITE_YEAR"]}.csv', low_memory=False, index_col=0)
location_df_kmeans

In [ ]:
location_df_kmeans.columns

In [ ]:
def construct_bout_metrics_from_classified_dets(fgroups_with_bouttags):
    """
    Reads in the dataframe of detected calls with bout tags.
    Uses these bout tags to create a new dataframe of bout metrics for the start and end times of each bout.
    Also includes the lowest frequency of a call within a bout as the lower bound for the bout
    and the highest frequency of a call within a bout as the upper bound frequency for the bout.
    Now, also included the number of detections captured within each bout.
    """

    location_df = fgroups_with_bouttags.copy()
    location_df.reset_index(drop=True, inplace=True)
    group_of_tagged_dets = location_df['freq_group'].unique().item()

    end_times_of_bouts = pd.to_datetime(location_df.loc[location_df['call_status']=='bout end', 'call_end_time'])
    start_times_of_bouts = pd.to_datetime(location_df.loc[location_df['call_status']=='bout start', 'call_start_time'])
    ref_end_times = location_df.loc[location_df['call_status']=='bout end', 'end_time_wrt_ref'].astype('float')
    ref_start_times = location_df.loc[location_df['call_status']=='bout start', 'start_time_wrt_ref'].astype('float')
    end_times = location_df.loc[location_df['call_status']=='bout end', 'end_time'].astype('float')
    start_times = location_df.loc[location_df['call_status']=='bout start', 'start_time'].astype('float')
    bout_starts = start_times_of_bouts.index
    bout_ends = end_times_of_bouts.index

    low_freqs = []
    high_freqs = []
    ref_time_cycle_start = []
    ref_time_cycle_end = []
    num_calls_per_bout = []
    input_files = []
    for i, bout_start in enumerate(bout_starts):
        bat_bout = location_df.iloc[bout_start:bout_ends[i]+1]
        bat_bout = bat_bout.loc[bat_bout['class']!='MADE-UP FOR DC INVESTIGATION']
        pass_low_freq = np.min(bat_bout['low_freq'])
        pass_high_freq = np.max(bat_bout['high_freq'])
        start_cycle = bat_bout['cycle_ref_time'].values[0]
        end_cycle = bat_bout['cycle_ref_time'].values[-1]
        bout_input_file = bat_bout['input_file'].values[0]
        num_calls = len(bat_bout)
        low_freqs += [pass_low_freq]
        high_freqs += [pass_high_freq]
        ref_time_cycle_start += [start_cycle]
        ref_time_cycle_end += [end_cycle]
        num_calls_per_bout += [num_calls]
        input_files += [bout_input_file]

    bout_metrics = pd.DataFrame()
    bout_metrics['start_time_of_bout'] = start_times_of_bouts.values
    bout_metrics['end_time_of_bout'] = end_times_of_bouts.values
    bout_metrics['start_time_wrt_ref'] = ref_start_times.values
    bout_metrics['end_time_wrt_ref'] = ref_end_times.values
    bout_metrics['start_time'] = start_times.values
    bout_metrics['end_time'] = end_times.values
    bout_metrics['low_freq'] = low_freqs
    bout_metrics['high_freq'] = high_freqs
    bout_metrics['freq_group'] = group_of_tagged_dets
    bout_metrics['input_file'] = input_files
    bout_metrics['cycle_ref_time_start'] = ref_time_cycle_start
    bout_metrics['cycle_ref_time_end'] = ref_time_cycle_end
    bout_metrics['number_of_dets'] = num_calls_per_bout
    bout_metrics['bout_duration'] = end_times_of_bouts.values - start_times_of_bouts.values
    bout_metrics['bout_duration_in_secs'] = bout_metrics['bout_duration'].apply(lambda x : x.total_seconds())

    return bout_metrics

def construct_bout_metrics_from_location_df_for_freqgroups(location_df):
    """
    Given a location summary with tagged bout markers, construct and concatenate together bout metrics for each group
    """

    bout_metrics = pd.DataFrame()
    for group in location_df['freq_group'].unique():
        if group != '':
            tagged_freq_dets = location_df.loc[location_df['freq_group']==group].copy()
            if not(tagged_freq_dets.empty):
                freqgroup_bout_metrics = construct_bout_metrics_from_classified_dets(tagged_freq_dets)
                if len(bout_metrics) > 0:
                    bout_metrics = pd.concat([bout_metrics, freqgroup_bout_metrics])
                else:
                    bout_metrics = freqgroup_bout_metrics.copy()

    return bout_metrics

In [ ]:
dc_applied_df = ss.simulate_dutycycle_on_detections(location_df_kmeans.copy(), data_params)
bout_params = bt.get_bout_params_from_location(dc_applied_df, data_params)
tagged_dets = bt.classify_bouts_in_detector_preds_for_freqgroups(dc_applied_df.copy(), bout_params)
bout_metrics = construct_bout_metrics_from_location_df_for_freqgroups(tagged_dets)

In [ ]:
bout_metrics.columns

In [ ]:
selected_group_kmeans = location_df_kmeans[pd.to_datetime(location_df_kmeans['input_file'], format="%Y%m%d_%H%M%S.WAV", exact=False)<=selection_date]
selected_group_kmeans

In [ ]:
def get_dropped_by_kmeans(thresh_file_df, all_file_kmeans_df):
    input_file_group_name = thresh_file_df.input_file.values[0]
    thresh_file_df = thresh_file_df.set_index('index_in_file')
    kmeans_file_df = all_file_kmeans_df[all_file_kmeans_df['input_file']==input_file_group_name]
    kmeans_file_df = kmeans_file_df.set_index('index_in_file')
    dropped_inds = sorted(list(set(thresh_file_df.index) - set(kmeans_file_df.index)))
    return thresh_file_df.loc[dropped_inds]

In [ ]:
data_params = dict()
data_params["site_tag"] = 'Foliage'
data_params["site_name"] = SITE_NAMES[data_params["site_tag"]]
data_params["type_tag"] = ''
data_params["detector_tag"] = 'bd2'
data_params["assembly_type"] = 'kmeans'

file_paths = get_file_paths(data_params)
file_paths['SITE_classes_file'] = f"{file_paths['SITE_classes_file'][:-4]}_raw.csv"
raw_location_df_filepath = Path(f'20241116__location_df_Foliage_kmeans_raw.csv')
if raw_location_df_filepath.is_file():
    location_df_kmeans_raw = pd.read_csv(raw_location_df_filepath, low_memory=False, index_col=0)
else:
    init_location_sum = actvt.assemble_initial_location_summary(file_paths) 
    init_location_sum.reset_index(inplace=True)
    init_location_sum.rename({'index':'index_in_file'}, axis='columns', inplace=True)
    location_df_kmeans_raw = add_frequency_groups_to_summary_using_kmeans(init_location_sum.copy(), file_paths, data_params, save=False)
    location_df_kmeans_raw.to_csv(raw_location_df_filepath)

file_paths = get_file_paths(data_params)
location_df_kmeans = pd.read_csv(f'{file_paths["SITE_folder"]}/{file_paths["detector_TYPE_SITE_YEAR"]}.csv', low_memory=False, index_col=0)

all_dropped_calls = location_df_kmeans_raw.groupby(by='input_file', group_keys=False).apply(lambda x : get_dropped_by_kmeans(x, location_df_kmeans))
test_df = all_dropped_calls.reset_index().reset_index()

In [ ]:
# all_dropped_calls = location_sum_kmeans.groupby(by='input_file', group_keys=False).apply(lambda x : get_dropped_by_median_freq_removal(x, location_sum_kmeans_remove_fp))
all_dropped_calls = location_df_kmeans_raw.groupby(by='input_file', group_keys=False).apply(lambda x : get_dropped_by_kmeans(x, location_df_kmeans))

In [ ]:
all_dropped_calls

In [ ]:
test_df = all_dropped_calls.reset_index().loc[:,['index_in_file', 'freq_group', 'start_time', 'end_time', 'low_freq', 'high_freq', 'input_file']]
test_df

In [ ]:
def get_section_of_call_in_file(detection, audio_file):
    fs = audio_file.samplerate

    call_dur = (detection['end_time'] - detection['start_time'])
    pad = min(min(detection['start_time'] - call_dur, 1795 - detection['end_time']), 0.006) / 3
    start = detection['start_time'] - call_dur - (3*pad)
    duration = (2 * call_dur) + (4*pad)

    audio_file.seek(int(fs*start))
    audio_seg = audio_file.read(int(fs*duration))

    length_of_section = call_dur + (2*pad)

    return audio_seg, length_of_section

In [ ]:
filesys = fsspec.filesystem('s3', anon=True, client_kwargs={'endpoint_url': 'https://sdsc.osn.xsede.org'})

In [ ]:
FREQUENCY_COLOR_MAPPINGS = {
                    'LF' : 'cyan',
                    'HF' : 'orange'
                        }

In [ ]:
import re

In [ ]:
cur_path = ''
for i in np.arange(0, len(test_df), 36):
    subset = test_df[i:min(i+36, len(test_df))].reset_index(drop=True).copy()
    matrix_side_length = int(np.ceil(len(subset)**0.5))
    plt.figure(figsize=(3*matrix_side_length,3*matrix_side_length))
    plt.rcParams.update({'font.size':12})

    for j, row in subset.iterrows():
        plt.subplot(matrix_side_length, matrix_side_length, j+1)
        file_path = '/'.join(Path(row['input_file']).parts[2:])
        cleaned_path = re.sub(r"(ubna_data_\d+)_mir", r"\1", file_path)
        osn_file_path = Path(f'bio230143-bucket01/{cleaned_path}')
        if cur_path!=osn_file_path:
            cur_path = osn_file_path
            file = filesys.open(path=cur_path)
            audio_file = sf.SoundFile(file)
            fs = audio_file.samplerate

        # audio_seg, length_of_section = get_section_of_call_in_file(row, audio_file)
        call_dur = (row['end_time'] - row['start_time'])
        pad = min(min(row['start_time'] - call_dur, 1795 - row['end_time']), 0.3) / 3
        start = row['start_time'] - call_dur - (3*pad)
        duration = (2 * call_dur) + (10*pad)

        audio_file.seek(int(fs*start))
        audio_seg = audio_file.read(int(fs*duration))

        plt.title(f'{osn_file_path.name}')
        plt.specgram(audio_seg, NFFT=256, cmap='jet', vmin=-60, vmax=0)

        file_df_orig = location_df_kmeans_raw[location_df_kmeans_raw['input_file']==row['input_file']]
        plot_dets = file_df_orig[(file_df_orig['start_time']>=start)&(file_df_orig['end_time']<=(start+duration))]
        ax = plt.gca()
        for k, det in plot_dets.iterrows():
            if det['start_time']==row['start_time']:
                rect = patches.Rectangle(((det['start_time'] - start)*(fs/2), det['low_freq']/(fs/2)), 
                                (det['end_time'] - det['start_time'])*(fs/2), (det['high_freq'] - det['low_freq'])/(fs/2), 
                                linewidth=3, edgecolor='red', facecolor='none', alpha=0.8)
            else:
                rect = patches.Rectangle(((det['start_time'] - start)*(fs/2), det['low_freq']/(fs/2)), 
                                (det['end_time'] - det['start_time'])*(fs/2), (det['high_freq'] - det['low_freq'])/(fs/2), 
                                linewidth=2, edgecolor=FREQUENCY_COLOR_MAPPINGS[det['freq_group']], facecolor='none', alpha=0.8)
            ax.add_patch(rect)

        plt.yticks(ticks=np.linspace(0, 1, 6), labels=np.linspace(0, fs/2000, 6).astype('int'))
        plt.ylabel("Frequency (kHz)", fontsize=14)
        plt.text(x=int(fs*0.001),y=0.85, s=f'{row["freq_group"]} det{i+j}', fontweight='bold', color='w')
        plt.xticks(ticks=np.linspace(0, duration*fs/2, 6), labels=np.round(np.linspace(start, start+duration, 6, dtype=float), 3), rotation=30)
        plt.xlabel("Time (s)")

    plt.tight_layout()
    plt.show()